In [6]:
import pickle
import numpy as np
from pathlib import Path

# ====== 只改这一个路径 ======
SEARCH_DIR = Path("/home/chendawww/workspace/rl-navibot/src/decision/rl_agent/saved_models/SAC")

pkl_files = list(SEARCH_DIR.rglob("*replay_buffer*"))
pkl_files = [f for f in pkl_files if f.stat().st_size > 1024]
pkl_files.sort(key=lambda f: f.stat().st_mtime, reverse=True)
REPLAY_PATH = pkl_files[0]

with open(REPLAY_PATH, "rb") as f:
    buf = pickle.load(f)

# 去掉中间那层多余的维度 (47999, 1, 2) → (47999, 2)
actions = buf.actions[:buf.pos, 0, :]
rewards = buf.rewards[:buf.pos]
dones   = buf.dones[:buf.pos]
obs     = buf.observations[:buf.pos, 0, :]

linear  = actions[:, 0]
angular = actions[:, 1]

print("=" * 60)
print("【全局动作统计】— 模型在整个训练后期的行为指纹")
print("=" * 60)
print(f"总条数: {len(actions)}")
print(f"线速度  均值={np.mean(linear):+.4f}  标准差={np.std(linear):.4f}  "
      f"范围=[{np.min(linear):+.4f}, {np.max(linear):+.4f}]")
print(f"角速度  均值={np.mean(angular):+.4f}  标准差={np.std(angular):.4f}  "
      f"范围=[{np.min(angular):+.4f}, {np.max(angular):+.4f}]")

lin_hist, _ = np.histogram(linear, bins=20)
lin_hist = lin_hist / lin_hist.sum()
max_bin = lin_hist.max()
print(f"线速度最大频率区间占比: {max_bin*100:.1f}%  "
      f"({'✅ 策略有明确输出偏好' if max_bin > 0.15 else '❌ 动作很分散'})")

ang_hist, _ = np.histogram(angular, bins=20)
ang_hist = ang_hist / ang_hist.sum()
ang_max_bin = ang_hist.max()
print(f"角速度最大频率区间占比: {ang_max_bin*100:.1f}%  "
      f"({'✅ 转向有偏好' if ang_max_bin > 0.15 else '❌ 转向很分散'})")
print()

# 分离 episode
done_indices = np.where(dones > 0.5)[0]
print(f"done 信号数: {len(done_indices)}")

episodes_to_show = min(3, len(done_indices) - 1)
if episodes_to_show <= 0:
    # 如果只有1个done，就拿最后一段数据
    done_indices = np.array([0])
    episodes_to_show = 1

for ep in range(episodes_to_show):
    end_idx = done_indices[-(ep + 1)]
    if ep + 2 <= len(done_indices):
        start_idx = done_indices[-(ep + 2)] + 1
    else:
        start_idx = 0

    ep_actions = actions[start_idx : end_idx + 1]
    ep_rewards = rewards[start_idx : end_idx + 1]
    ep_obs     = obs[start_idx : end_idx + 1]
    ep_len = len(ep_actions)
    ep_total_reward = np.sum(ep_rewards)

    ep_lin = ep_actions[:, 0]
    ep_ang = ep_actions[:, 1]

    print()
    print("=" * 60)
    print(f"【Episode 最近第 {ep+1} 个】步数={ep_len}  总奖励={ep_total_reward:.2f}")
    print("=" * 60)

    # 动作时间线
    print(f"{'步数':>6s}  {'线速度':>9s}  {'角速度':>9s}  {'obs均值':>8s}")
    print("-" * 40)
    sample_step = max(1, ep_len // 30)
    for i in range(0, ep_len, sample_step):
        print(f"{i:>6d}  {ep_actions[i,0]:>+9.4f}  {ep_actions[i,1]:>+9.4f}  {np.mean(ep_obs[i]):>8.4f}")

    # 诊断指标
    ang_sign_changes = np.sum(np.diff(np.sign(ep_ang)) != 0)
    ang_change_rate = ang_sign_changes / max(1, ep_len - 1)
    moving_ratio = np.mean(ep_lin > 0.02)

    # 线速度为正（前进）且角速度较小的步数占比 → "稳定直线行驶"
    straight_ratio = np.mean((ep_lin > 0.05) & (np.abs(ep_ang) < 0.3))

    # 角速度绝对值的均值 → 转向强度
    mean_abs_ang = np.mean(np.abs(ep_ang))

    print(f"\n  ┌─ 诊断指标 ──────────────────────────────────┐")
    print(f"  │ 角速度翻转率:     {ang_change_rate:>6.1%}  "
          f"{'✅果断' if ang_change_rate < 0.3 else '❌摇摆'}"
          f"                       │")
    print(f"  │ 线速度>0.02比例:  {moving_ratio:>6.1%}  "
          f"{'✅在动' if moving_ratio > 0.6 else '❌停滞'}"
          f"                       │")
    print(f"  │ 稳定直行比例:     {straight_ratio:>6.1%}  "
          f"{'✅会走直线' if straight_ratio > 0.3 else '❌不会直行'}"
          f"                   │")
    print(f"  │ 平均转向强度:     {mean_abs_ang:>6.3f}  "
          f"{'✅适度' if 0.1 < mean_abs_ang < 1.0 else '⚠️极端'}"
          f"                       │")
    print(f"  └────────────────────────────────────────────┘")

【全局动作统计】— 模型在整个训练后期的行为指纹
总条数: 47999
线速度  均值=+0.8119  标准差=0.3588  范围=[-0.9999, +1.0000]
角速度  均值=-0.0056  标准差=0.3996  范围=[-0.9998, +0.9999]
线速度最大频率区间占比: 64.5%  (✅ 策略有明确输出偏好)
角速度最大频率区间占比: 11.0%  (❌ 转向很分散)

done 信号数: 574

【Episode 最近第 1 个】步数=124  总奖励=367.52
    步数        线速度        角速度     obs均值
----------------------------------------
     0    -0.0535    -0.9371    0.2518
     4    +0.5583    -0.9676    0.1728
     8    +0.8823    -0.9389    0.2174
    12    +0.8598    -0.6478    0.2180
    16    +0.3818    -0.4594    0.2260
    20    +0.9131    -0.3195    0.2632
    24    +0.9582    +0.0552    0.3455
    28    +0.9629    +0.2945    0.3315
    32    +0.9942    +0.0545    0.3039
    36    +0.6360    +0.1729    0.3609
    40    +0.9434    +0.0942    0.3266
    44    +0.9685    +0.1096    0.3406
    48    +0.9326    -0.2143    0.3491
    52    +0.9837    +0.2688    0.3236
    56    +0.9750    -0.4683    0.3566
    60    +0.8942    +0.0650    0.3222
    64    +0.7264    -0.2931    0.3361
   

In [4]:
import pickle
import numpy as np
from pathlib import Path

SEARCH_DIR = Path("/home/chendawww/workspace/rl-navibot/src/decision/rl_agent/saved_models/SAC")

pkl_files = list(SEARCH_DIR.rglob("*replay_buffer*"))
pkl_files = [f for f in pkl_files if f.stat().st_size > 1024]

if not pkl_files:
    print("❌ 没找到 replay buffer 文件！")
    exit()

pkl_files.sort(key=lambda f: f.stat().st_mtime, reverse=True)
REPLAY_PATH = pkl_files[0]
print(f"✅ 加载: {REPLAY_PATH}")

with open(REPLAY_PATH, "rb") as f:
    buf = pickle.load(f)

print(f"类型: {type(buf)}")
print(f"当前数据量: {buf.pos}")
print(f"是否已满: {buf.full}")
print(f"动作空间: {buf.action_space}  形状: {buf.action_space.shape}")
print(f"观测空间: {buf.observation_space}  形状: {buf.observation_space.shape}")

actions = buf.actions[:buf.pos]
rewards = buf.rewards[:buf.pos]
dones   = buf.dones[:buf.pos]
obs     = buf.observations[:buf.pos]

print(f"\nactions 形状: {actions.shape}")
print(f"obs 形状:     {obs.shape}")

# 先看一下 obs 的第一帧长什么样
if obs.ndim == 3:  # (N, 1, obs_dim) SB3 有时会包一层
    obs_squeeze = obs[:, 0, :]
else:
    obs_squeeze = obs

print(f"\n首帧 obs 均值: {np.mean(obs_squeeze[0]):.4f}  范围: [{np.min(obs_squeeze[0]):.4f}, {np.max(obs_squeeze[0]):.4f}]")
print(f"尾帧 obs 均值: {np.mean(obs_squeeze[-1]):.4f}  范围: [{np.min(obs_squeeze[-1]):.4f}, {np.max(obs_squeeze[-1]):.4f}]")



✅ 加载: /home/chendawww/workspace/rl-navibot/src/decision/rl_agent/saved_models/SAC/sac_nav_model_replay_buffer_148000_steps.pkl
类型: <class 'stable_baselines3.common.buffers.ReplayBuffer'>
当前数据量: 47999
是否已满: True
动作空间: Box([-0.22 -1.5 ], [0.22 1.5 ], (2,), float32)  形状: (2,)
观测空间: Box(-1.0, 1.0, (38,), float32)  形状: (38,)

actions 形状: (47999, 1, 2)
obs 形状:     (47999, 1, 38)

首帧 obs 均值: 0.3197  范围: [-0.1092, 1.0000]
尾帧 obs 均值: 0.3601  范围: [-0.0058, 1.0000]
